In [ ]:
import pandas as pd
import numpy as np
 
 
def build_feature_vectors(combined, daily_indicators):
    combined['day'] = combined['date'].dt.date
    combined['hour'] = combined['date'].dt.hour
    combined['dayofweek'] = combined['date'].dt.dayofweek  # 0=Monday, 6=Sunday
 
    
    logon_only = combined[combined['source'] == 'logon']
    first_login_hour = (
        logon_only.groupby(['user', 'day'])['hour']
        .min()
        .reset_index(name='first_login_hour')
    )
 
    
    last_activity_hour = (
        combined.groupby(['user', 'day'])['hour']
        .max()
        .reset_index(name='last_activity_hour')
    )
 
    
    day_of_week = (
        combined.groupby(['user', 'day'])['dayofweek']
        .first()
        .reset_index(name='dayofweek')
    )
    day_of_week['is_weekend'] = (day_of_week['dayofweek'] >= 5).astype(int)
    day_of_week = day_of_week.drop(columns='dayofweek')
 
    )
    after_hours = combined[
        (combined['hour'] < 8) | (combined['hour'] >= 19)
    ]
    after_hours_count = (
        after_hours.groupby(['user', 'day'])
        .size()
        .reset_index(name='after_hours_count')
    )
 
    
    features = daily_indicators.copy()
    features = features.merge(first_login_hour, on=['user', 'day'], how='left')
    features = features.merge(last_activity_hour, on=['user', 'day'], how='left')
    features = features.merge(day_of_week, on=['user', 'day'], how='left')
    features = features.merge(after_hours_count, on=['user', 'day'], how='left')
 
    
    features['after_hours_count'] = features['after_hours_count'].fillna(0).astype(int)
    features['first_login_hour'] = features['first_login_hour'].fillna(-1)
    features['is_weekend'] = features['is_weekend'].fillna(0).astype(int)
 
    return features
 
 

feature_vectors = build_feature_vectors(combined, daily_indicators)
print(f"Feature vectors shape: {feature_vectors.shape}")
print(feature_vectors.head(10))

In [ ]:


#Analyze login hour distributions across all users. Flag users who regularly log in outside business hours.


import pandas as pd
import matplotlib.pyplot as plt


def analyze_temporal_patterns(combined, feature_vectors):
    logon_only = combined[combined['source'] == 'logon']
    logon_only = logon_only.copy()
    logon_only['hour'] = logon_only['date'].dt.hour

    
    plt.figure(figsize=(12, 4))
    logon_only['hour'].hist(bins=24, edgecolor='black')
    plt.title('Login Hour Distribution (All Users)')
    plt.xlabel('Hour of Day')
    plt.ylabel('Number of Logins')
    plt.xticks(range(0, 24))
    plt.tight_layout()
    plt.savefig('login_hour_distribution.png')
    plt.show()
    print("Saved: login_hour_distribution.png")

    # Flag users who frequently log in outside business hours (before 8 or after 19)
    after_hours_logins = logon_only[
        (logon_only['hour'] < 8) | (logon_only['hour'] >= 19)
    ]
    after_hours_per_user = (
        after_hours_logins.groupby('user')
        .size()
        .reset_index(name='after_hours_login_count')
        .sort_values('after_hours_login_count', ascending=False)
    )

    print("\nTop 10 users with most after-hours logins:")
    print(after_hours_per_user.head(10))

    # 3. Weekend activity per user
    weekend_activity = feature_vectors[feature_vectors['is_weekend'] == 1]
    weekend_per_user = (
        weekend_activity.groupby('user')
        .size()
        .reset_index(name='weekend_days_active')
        .sort_values('weekend_days_active', ascending=False)
    )

    print("\nTop 10 users with most weekend activity:")
    print(weekend_per_user.head(10))

    return after_hours_per_user, weekend_per_user


after_hours_per_user, weekend_per_user = analyze_temporal_patterns(combined, feature_vectors)

In [ ]:

#Challenge 7: Sequence-Based Activity Tracking



import pandas as pd


def build_activity_sequences(combined):
    combined['day'] = combined['date'].dt.date

    # 1. Build daily activity sequence per user
   
    combined['event_label'] = combined['source'] + ':' + combined['activity'].astype(str)

    sequences = (
        combined.sort_values(['user', 'date'])
        .groupby(['user', 'day'])['event_label']
        .apply(list)
        .reset_index(name='activity_sequence')
    )

    #  Flag suspicious patterns:
    #    - Device connect followed by large http burst
    #    - Activity outside business hours
    #    - Very long sequences (unusually active day)

    def flag_suspicious(seq):
        flags = []
        seq_str = ' '.join(seq)

        # Device used AND lots of http activity same day
        has_device = any('device' in e for e in seq)
        http_count = sum(1 for e in seq if 'http' in e)
        if has_device and http_count > 50:
            flags.append('device_and_high_http')

        
        if len(seq) > 100:
            flags.append('high_volume_day')

        
        if 'logon:Logon' in seq_str and len(seq) > 5:
            flags.append('active_session')

        return flags if flags else ['normal']

    sequences['flags'] = sequences['activity_sequence'].apply(flag_suspicious)
    sequences['is_suspicious'] = sequences['flags'].apply(
        lambda x: 0 if x == ['normal'] else 1
    )

    suspicious = sequences[sequences['is_suspicious'] == 1]
    print(f"Total user-days: {len(sequences)}")
    print(f"Suspicious user-days: {len(suspicious)}")
    print("\nSample suspicious sequences:")
    print(suspicious[['user', 'day', 'flags']].head(10))

    return sequences


sequences = build_activity_sequences(combined)

In [ ]:

#Challenge 8: User Risk Scoring


import pandas as pd
import numpy as np


def compute_risk_scores(feature_vectors, baseline_profiles):
    # 1. Merge daily features with baseline profiles
    df = feature_vectors.merge(baseline_profiles, on='user', how='left')

    indicator_cols = ['login_count', 'device_event_count',
                      'http_event_count', 'unique_pc_count']

    
    for col in indicator_cols:
        mean_col = f"{col}_mean"
        std_col = f"{col}_std"
        threshold = df[mean_col] + 2 * df[std_col]
        df[f"{col}_anomaly"] = (df[col] > threshold).astype(int)

    # Also flag after-hours activity as a risk signal
    df['after_hours_anomaly'] = (df['after_hours_count'] > 0).astype(int)
    df['weekend_anomaly'] = df['is_weekend']

    # Count how many anomaly flags fired on each day
    anomaly_cols = [f"{col}_anomaly" for col in indicator_cols] + \
                   ['after_hours_anomaly', 'weekend_anomaly']
    df['daily_anomaly_score'] = df[anomaly_cols].sum(axis=1)

    # Cumulative risk score per user:
    #    Sum of all daily anomaly scores, normalized to 0-100
    risk_scores = (
        df.groupby('user')['daily_anomaly_score']
        .sum()
        .reset_index(name='raw_risk_score')
    )

    max_score = risk_scores['raw_risk_score'].max()
    risk_scores['risk_score'] = (
        (risk_scores['raw_risk_score'] / max_score) * 100
    ).round(2)

    risk_scores = risk_scores.sort_values('risk_score', ascending=False)

    print("Top 20 highest risk users:")
    print(risk_scores.head(20))

    return df, risk_scores


daily_with_anomalies, risk_scores = compute_risk_scores(feature_vectors, baseline_profiles)